# 13 — RAG ENSEA complet de bout en bout

Charge les index validés et expose une fonction unique pour les questions simples, Facebook, exhaustives et conversationnelles.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


In [ ]:
import time
import pandas as pd
from IPython.display import Markdown, display
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
print('Embedding :', config.embeddings.model_name)
print('Reranker :', config.reranker.model_name)
print('Générateur :', config.generation.model_name)
print('Résultats normaux/exhaustifs :', config.retrieval.rerank_top_k, '/', config.retrieval.exhaustive_max_results)


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    from rag_pipeline import RAGPipeline
    started = time.perf_counter()
    pipeline = RAGPipeline(config)
    print('Pipeline chargé en', round(time.perf_counter() - started, 2), 's')
    print('Index institutionnel :', pipeline.store.dense.count())
    print('Index economique :', pipeline.economic_store.dense.count() if pipeline.economic_store else 0)
    print('Index Facebook :', pipeline.facebook_store.dense.count() if pipeline.facebook_store else 0)
else:
    print('Chargement désactivé. Fermez les autres kernels puis passez RUN_PIPELINE à True.')


In [ ]:
conversation_history = []

def ask_ensea(question, *, remember=True):
    started = time.perf_counter()
    response = pipeline.ask(question, conversation_history=conversation_history)
    display(Markdown(response.answer))
    display(pd.Series({
        'route': pipeline.retriever.last_decision.route,
        'contextualized': pipeline.last_contextualized,
        'expanded_retrieval': pipeline.last_expanded_retrieval,
        'retrieval_seconds': pipeline.retriever.last_timings.get('total_seconds', 0),
        'generation_seconds': pipeline.generator.last_generation_stats.get('total_seconds', 0),
        'total_seconds': time.perf_counter() - started,
        'cache_hit': pipeline.generator.last_generation_stats.get('cache_hit', False),
        'citations': len(response.citations),
    }).to_frame('valeur'))
    if remember:
        conversation_history.extend([
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': response.answer},
        ])
        del conversation_history[:-10]
    return response


In [ ]:
RUN_EXAMPLES = False
if RUN_EXAMPLES:
    ask_ensea("Quels sont les finalistes du concours d'éloquence du CDR de cette année ?")
    ask_ensea('La liste est-elle exhaustive ? Je veux la liste complète.')
else:
    print('Exemples désactivés. Activez RUN_PIPELINE puis RUN_EXAMPLES.')


## Notebook final officiel

Ce notebook termine le parcours pédagogique. Le fichier `rag_ensea.ipynb` à la racine reste le notebook final d'usage courant, tandis que celui-ci montre explicitement comment toutes les étapes précédentes s'assemblent.
